# TalknShop Research Metrics Analysis Template

This notebook computes the core metrics proposed in the paper and generates paper-ready plots.

Expected input files (CSV):
- `documentation/metrics_template.csv` (task outcomes + SUS answers)
- Optional: `documentation/result_quality.csv` (for precision/recall)
- Optional: `documentation/latency_breakdown.csv` (for latency chart)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from scipy import stats

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

ROOT = Path.cwd()
if (ROOT / 'documentation').exists():
    DOCS_DIR = ROOT / 'documentation'
else:
    DOCS_DIR = ROOT

METRICS_CSV = DOCS_DIR / 'metrics_template.csv'
RESULT_QUALITY_CSV = DOCS_DIR / 'result_quality.csv'
LATENCY_CSV = DOCS_DIR / 'latency_breakdown.csv'

print('Docs dir:', DOCS_DIR)
print('Metrics file exists:', METRICS_CSV.exists())

In [ ]:
df = pd.read_csv(METRICS_CSV)

for c in ['start_ts', 'end_ts']:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce', utc=True)

# If task_time_sec is missing, compute from timestamps.
if 'task_time_sec' not in df.columns and {'start_ts', 'end_ts'}.issubset(df.columns):
    df['task_time_sec'] = (df['end_ts'] - df['start_ts']).dt.total_seconds()

df.head()

## SUS Score Calculation

SUS scoring rule:
- Odd questions (1,3,5,7,9): score - 1
- Even questions (2,4,6,8,10): 5 - score
- Sum all adjusted values and multiply by 2.5

In [ ]:
sus_cols = [f'sus_q{i}' for i in range(1, 11)]

if set(sus_cols).issubset(df.columns):
    odd_cols = [f'sus_q{i}' for i in [1, 3, 5, 7, 9]]
    even_cols = [f'sus_q{i}' for i in [2, 4, 6, 8, 10]]

    adjusted = pd.DataFrame(index=df.index)
    adjusted[odd_cols] = df[odd_cols].apply(pd.to_numeric, errors='coerce') - 1
    adjusted[even_cols] = 5 - df[even_cols].apply(pd.to_numeric, errors='coerce')
    df['sus_score'] = adjusted.sum(axis=1) * 2.5
else:
    df['sus_score'] = np.nan

df[['participant_id', 'group', 'sus_score']].dropna().drop_duplicates().head()

## Core Metrics (Descriptive Stats)

In [ ]:
summary_time = (
    df.groupby(['group', 'task_type'], dropna=False)['task_time_sec']
      .agg(['count', 'mean', 'median', 'std'])
      .reset_index()
)

summary_success = (
    df.groupby(['group', 'task_type'], dropna=False)['completed']
      .mean()
      .reset_index(name='success_rate')
)

summary_purchase = (
    df.groupby(['group', 'task_type'], dropna=False)['purchase_made']
      .mean()
      .reset_index(name='purchase_rate')
)

summary_turns = (
    df.groupby(['group', 'task_type'], dropna=False)['conversation_turns']
      .agg(['mean', 'median'])
      .reset_index()
)

display(summary_time)
display(summary_success)
display(summary_purchase)
display(summary_turns)

## Statistical Tests (TalknShop vs Traditional)

In [ ]:
def compare_groups_continuous(data, metric):
    a = pd.to_numeric(data.loc[data['group'] == 'talknshop', metric], errors='coerce').dropna()
    b = pd.to_numeric(data.loc[data['group'] == 'traditional', metric], errors='coerce').dropna()
    if len(a) < 2 or len(b) < 2:
        return {'metric': metric, 'test': 'insufficient data'}

    # Welch's t-test (robust to unequal variances)
    t_stat, p_val = stats.ttest_ind(a, b, equal_var=False)

    # Cohen's d
    pooled_sd = np.sqrt(((a.std(ddof=1) ** 2) + (b.std(ddof=1) ** 2)) / 2)
    d = (a.mean() - b.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan

    return {
        'metric': metric,
        'talknshop_mean': a.mean(),
        'traditional_mean': b.mean(),
        't_stat': t_stat,
        'p_value': p_val,
        'cohens_d': d,
    }

tests = [
    compare_groups_continuous(df, 'task_time_sec'),
    compare_groups_continuous(df, 'conversation_turns'),
    compare_groups_continuous(df, 'clarification_rounds'),
    compare_groups_continuous(df, 'sus_score'),
]

pd.DataFrame(tests)

## Paper Plot 1: Task Completion Time by Task Type

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df, x='task_type', y='task_time_sec', hue='group', errorbar='sd')
ax.set_title('Task Completion Time by Task Type')
ax.set_xlabel('Task Type')
ax.set_ylabel('Time (sec)')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Paper Plot 2: SUS Distribution

In [ ]:
sus_df = df[['participant_id', 'group', 'sus_score']].dropna().drop_duplicates()

if not sus_df.empty:
    plt.figure(figsize=(7, 4))
    ax = sns.boxplot(data=sus_df, x='group', y='sus_score')
    ax.axhline(68, color='red', linestyle='--', linewidth=1, label='SUS=68 benchmark')
    ax.set_title('SUS Score Distribution by Group')
    ax.set_xlabel('Group')
    ax.set_ylabel('SUS Score (0-100)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No SUS data available.')

## Optional: Precision@k and Recall@k

Expected `result_quality.csv` columns:
- `task_id`, `returned_rank`, `is_relevant`
- Optional grouping columns: `group`, `product_category`

In [ ]:
def precision_recall_at_k(dfq, k=5):
    topk = dfq.nsmallest(k, 'returned_rank')
    rel_topk = topk['is_relevant'].sum()
    total_rel = dfq['is_relevant'].sum()
    precision = rel_topk / k if k > 0 else np.nan
    recall = rel_topk / total_rel if total_rel > 0 else np.nan
    return precision, recall

if RESULT_QUALITY_CSV.exists():
    rq = pd.read_csv(RESULT_QUALITY_CSV)
    rq['returned_rank'] = pd.to_numeric(rq['returned_rank'], errors='coerce')
    rq['is_relevant'] = pd.to_numeric(rq['is_relevant'], errors='coerce').fillna(0)

    rows = []
    for task_id, grp in rq.groupby('task_id'):
        p5, r5 = precision_recall_at_k(grp, k=5)
        rows.append({'task_id': task_id, 'precision_at_5': p5, 'recall_at_5': r5})

    pr = pd.DataFrame(rows)
    display(pr.head())
    print('Mean Precision@5:', pr['precision_at_5'].mean())
    print('Mean Recall@5:', pr['recall_at_5'].mean())
else:
    print('Optional file not found:', RESULT_QUALITY_CSV)

## Optional: Latency Breakdown Plot

Expected `latency_breakdown.csv` columns:
- `group`, `task_type`, `ws_ms`, `llm_ms`, `api_ms`, `ranking_ms`, `total_ms`

In [ ]:
if LATENCY_CSV.exists():
    lat = pd.read_csv(LATENCY_CSV)
    comp_cols = ['ws_ms', 'llm_ms', 'api_ms', 'ranking_ms']
    lat_grouped = lat.groupby('task_type', dropna=False)[comp_cols].mean().reset_index()

    lat_grouped = lat_grouped.set_index('task_type')
    lat_grouped.plot(kind='bar', stacked=True, figsize=(10, 5))
    plt.title('Latency Breakdown by Task Type (Mean ms)')
    plt.xlabel('Task Type')
    plt.ylabel('Latency (ms)')
    plt.xticks(rotation=25, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Optional file not found:', LATENCY_CSV)

## Next Steps

1. Replace sample rows in `metrics_template.csv` with real study data.
2. Add `result_quality.csv` and `latency_breakdown.csv` when available.
3. Export summary tables and figures for your paper.
4. Report p-values, effect sizes, and confidence intervals in the Results section.